# Z3-Rust-08 : Ordonnancement de taches (Job-Shop Scheduling)

[← Z3-Python-08 (ordonnancement)](Z3-08-Ordonnancement-Python.ipynb) | [Serie Z3-API](README.md) | [Jumeau Python](Z3-08-Ordonnancement-Python.ipynb) | [Z3-Python-09 (enigma d'Einstein) →](Z3-09-Enigme-Einstein-Python.ipynb)

## Le probleme

Le **job-shop scheduling** est un probleme canonique d'optimisation combinatoire : on dispose de $m$ machines et de $n$ jobs, chaque job etant une **sequence d'operations** (une par machine) de durees donnees. On veut trouver un ordonnancement (date de debut de chaque operation) qui :

1. **respecte l'ordre** des operations de chaque job (precedence),
2. **n'utilise chaque machine qu'une fois a la fois** (exclusion disjonctive),
3. **minimise le makespan** $C_{\max}$ (la date de fin de la derniere operation).

Ce probleme est **NP-difficile** : il n'existe pas d'algorithme polynomial connu, et une approche gloutonne (heuristique) rate souvent l'optimum. C'est typiquement la ou un solveur SMT comme Z3 **brille** : on decrit les contraintes (precedence + exclusion) et l'objectif (minimiser $C_{\max}$), et `Optimize` trouve l'ordonnancement optimal.

**Ce notebook montre** : (a) une heuristique gloutonne FIFO sous-optimale, (b) la modelisation Z3 avec la contrainte disjonctive `Or(... , ...)`, (c) l'optimum trouve par `Optimize.minimize`, et (d) un diagramme de Gantt comparant les deux. Il est le **jumeau Rust** de [Z3-08-Ordonnancement-Python](Z3-08-Ordonnancement-Python.ipynb) : meme instance, meme modele, memes grandeurs mesurees.

## Ce qui change en Rust

Le modele est identique au jumeau Python ; seule l'API change. Trois differences structurantes :

| Aspect | Python (`z3-solver`) | Rust (crate `z3`) |
|---|---|---|
| Solveur | `z3.Optimize()` | `Optimize::new()` |
| Disjonction | `z3.Or(a, b)` | `Bool::or(&[&a, &b])` |
| Somme / difference | `a + b` | `Int::add(&[a, b])` (fonction variadique) |
| Modele | `opt.model()[v]` | `opt.get_model()` puis `model.eval(&v, true)` |

Le crate utilise ici est **`z3` 0.23.0, sans la feature `gh-release`** : la bibliotheque Z3 est liee dynamiquement a celle du poste, la feature statique n'existant pas pour la plateforme Windows du siege. La version de la bibliotheque effective est affichee par la premiere cellule.

## Source

Ce notebook s'inscrit dans la serie ancree par **Yurichev, Dennis — *SAT/SMT by Example* (edition publique du 12 fevrier 2023, 666 pages)**, §22.8 Job Shop Scheduling (p. 633), §22.8.1 My first version (p. 633) et §22.8.2 MaxSMT version by Chad Brewbaker (p. 635) — le probleme est pose a l'identique (jobs, machines, operations, `makespan` minimise, diagramme de Gantt), avec les instances 3×3 puis 10 machines. Le carnet reprend l'instance 3 jobs × 2 machines et la comparaison glouton vs optimum. Gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16,45 Mo, magic bytes `%PDF`). Pages citees d'apres la pagination imprimee du fichier depose. Voir [Z3-API/README.md — Ouvrage de reference](README.md#ouvrage-de-reference) pour l'index chapitre de la serie.

In [1]:
// Imports : le crate z3 (solveur SMT) et l'API de modelisation.
// Le crate est lie dynamiquement a la bibliotheque Z3 installee sur le poste.
:dep z3 = { version = "0.23.0" }

use z3::ast::{Ast, Bool, Int};

println!("Imports OK : z3");
println!("z3 (lib) version : {}", z3::full_version());

Imports OK : z3


z3 (lib) version : 5.3.3.0 430d19e697e2ec9bad79ff210d41b4a7a5fcfa3e 430d19e


## Instance de reference : 3 jobs × 2 machines

On encode l'instance comme un tableau de couples : chaque job est la liste ordonnee de ses operations, chaque operation etant un couple `(machine, duree)`.

| Job | Operation 1 | Operation 2 |
|-----|-------------|-------------|
| J0  | M0 (3h)     | M1 (3h)     |
| J1  | M1 (2h)     | M0 (2h)     |
| J2  | M0 (2h)     | M1 (2h)     |

Notez que J0 et J2 commencent par M0, mais J1 commence par M1 : les jobs n'ont pas tous le meme parcours.

In [2]:
// Instance : 3 jobs x 2 machines (M0, M1).
// Format : JOBS[j] = [(machine, duree), (machine, duree)] dans l'ordre des operations.
const MACHINES: [&str; 2] = ["M0", "M1"];
// Le solveur est genererique : une operation est un couple (machine, duree)
// et le nombre d'operations par job est porte par un Vec. En Python, la meme
// information est une liste et la generalisation est gratuite ; en Rust, la
// longueur d'un tableau est dans le type -- c'est la seule generalisation que
// le portage exige (l'algorithme, lui, boucle deja sur ops.len()).
let JOBS: Vec<(&str, Vec<(&str, i64)>)> = vec![
    ("J0", vec![("M0", 3), ("M1", 3)]),
    ("J1", vec![("M1", 2), ("M0", 2)]),
    ("J2", vec![("M0", 2), ("M1", 2)]),
];

println!("Instance : {} jobs x {} machines", JOBS.len(), MACHINES.len());
for (j, ops) in JOBS.iter() {
    let parcours: Vec<String> = ops.iter().map(|(m, d)| format!("{}({}h)", m, d)).collect();
    println!("  {} : {}", j, parcours.join(" puis "));
}
// evcxr afficherait sinon la valeur unitaire () du bloc : on neutralise
// l'affichage pour que la sortie reste comparable a celle du jumeau Python.
let _ = &JOBS;


Instance : 3 jobs x 2 machines


  J0 : M0(3h) puis M1(3h)


  J1 : M1(2h) puis M0(2h)


  J2 : M0(2h) puis M1(2h)


## Pourquoi ce probleme est-il difficile ?

Le job-shop scheduling est **NP-difficile** (Garey & Johnson, 1976) : des que le nombre de jobs et de machines croit, l'espace des ordonnancements possibles explose, car il faut choisir un **ordre des operations sur chaque machine** parmi tous les ordres compatibles. Notre instance 3 jobs × 2 machines reste complètement lisible — c'est le but pedagogique — mais la meme structure à 10 jobs × 10 machines defie les approches exactes depuis les annees 1960 (l'instance canonique FT10 de Fisher & Thompson n'a ete close qu'en 1989).

Deux bornes encadrent deja l'espace des solutions :

- **Borne inferieure de charge** : chaque machine doit absorber la somme des durees des operations qui la traversent, soit 7 h pour M0 (3 + 2 + 2) et 7 h pour M1 (3 + 2 + 2). Aucun ordonnancement ne peut finir avant t = 7.
- **Horizon sequentiel** : si tout etait execute a la suite, la date de fin serait la somme de toutes les durees : 6 + 4 + 4 = **14 h**. C'est exactement la valeur que donnera le glouton ci-dessous — et la borne superieure que nous fournirons a Z3.

La question devient : peut-on descendre sous 14 h en **entrelaçant** les jobs ? C'est precisement ce que la contrainte disjonctive va exprimer.

## Approche gloutonne : ordonnancement FIFO

Avant Z3, essayons une **heuristique simple** : on schedule les jobs dans l'ordre (J0, puis J1, puis J2), et pour chaque job on place chaque operation **le plus tot possible** (des que la machine est libre et que l'operation precedente du job est terminee).

Cette strategie FIFO est rapide mais **myope** : elle ne regarde pas plus loin que le job courant, et peut laisser une machine inutilisee la ou un autre ordre aurait mieux equilibre la charge.

In [3]:
// Ordonnancement glouton FIFO : chaque job dans l'ordre, chaque operation ASAP.
// Une machine n'est jamais liberee avant la fin de l'operation qui l'occupe.
fn greedy_fifo(jobs: &[(&str, Vec<(&str, i64)>)], machine_names: &[&str])
    -> (Vec<Vec<(String, i64, i64)>>, i64) {
    let mut machine_free: Vec<(String, i64)> =
        machine_names.iter().map(|m| (m.to_string(), 0)).collect();
    let mut schedule: Vec<Vec<(String, i64, i64)>> = Vec::new();
    for (_j, ops) in jobs.iter() {
        let mut job_ops: Vec<(String, i64, i64)> = Vec::new();
        for (m, d) in ops.iter() {
            // L'operation du job ne peut pas demarrer avant la fin de la precedente.
            let job_ready = job_ops.last().map(|(_, s, dur)| s + dur).unwrap_or(0);
            let start = match machine_free.iter_mut().find(|(name, _)| name == m) {
                Some(slot) => {
                    let s = std::cmp::max(slot.1, job_ready);
                    slot.1 = s + d;
                    s
                }
                None => job_ready,
            };
            job_ops.push((m.to_string(), start, *d));
        }
        schedule.push(job_ops);
    }
    let cmax = schedule.iter().flatten().map(|(_, s, d)| s + d).max().unwrap_or(0);
    (schedule, cmax)
}

let (greedy_schedule, greedy_cmax) = greedy_fifo(&JOBS, &MACHINES);
println!("Glouton FIFO : Cmax = {}h", greedy_cmax);
for (ji, (j, _ops)) in JOBS.iter().enumerate() {
    let ops: Vec<String> = greedy_schedule[ji]
        .iter()
        .map(|(m, s, d)| format!("{}[{},{}]", m, s, s + d))
        .collect();
    println!("  {} : {}", j, ops.join(", "));
}
// evcxr afficherait sinon la valeur unitaire () du bloc : on neutralise
// l'affichage pour que la sortie reste comparable a celle du jumeau Python.
let _ = &greedy_schedule;


Glouton FIFO : Cmax = 14h


  J0 : M0[0,3], M1[3,6]


  J1 : M1[6,8], M0[8,10]


  J2 : M0[10,12], M1[12,14]


## Modelisation Z3 : variables et contraintes

Le passage au declaratif. On introduit une variable entiere $s_{j,k}$ = date de debut de l'operation $k$ du job $j$, plus une variable $C_{\max}$ pour le makespan. Les contraintes se traduisent directement :

1. **Precedence intra-job** : l'operation $k$ commence apres la fin de l'operation $k{-}1$ :  $s_{j,k} \ge s_{j,k-1} + d_{j,k-1}$.
2. **Exclusion disjonctive** (le coeur du modele) : pour deux operations $a$ et $b$ sur la **meme** machine, l'une doit finir avant que l'autre commence :  $s_a + d_a \le s_b$ **OU** $s_b + d_b \le s_a$. C'est cette disjonction `Or(...)` (non-lineaire, difficile pour un simple parcours) qui fait toute la valeur du solveur.
3. **Objectif** : minimiser $C_{\max} = \max_{j,k}(s_{j,k} + d_{j,k})$, via `Optimize.minimize`.

La borne superieure sure $C_{\max} \le$ somme de toutes les durees (cas ou tout serait sequentialise) garantit un domaine fini au solveur.

En Rust, la somme `s + d` s'ecrit `Int::add(&[s, d])` (le crate expose les operateurs arithmetiques comme des fonctions variadiques), et la disjonction `Or(a, b)` s'ecrit `Bool::or(&[&a, &b])`.

In [4]:
use z3::ast::{Ast, Bool, Int};
use z3::{Optimize, SatResult};

const MACHINES: [&str; 2] = ["M0", "M1"];
// Le solveur est genererique : une operation est un couple (machine, duree)
// et le nombre d'operations par job est porte par un Vec. En Python, la meme
// information est une liste et la generalisation est gratuite ; en Rust, la
// longueur d'un tableau est dans le type -- c'est la seule generalisation que
// le portage exige (l'algorithme, lui, boucle deja sur ops.len()).
let JOBS: Vec<(&str, Vec<(&str, i64)>)> = vec![
    ("J0", vec![("M0", 3), ("M1", 3)]),
    ("J1", vec![("M1", 2), ("M0", 2)]),
    ("J2", vec![("M0", 2), ("M1", 2)]),
];
// Modele Z3 du job-shop : precedence intra-job, exclusion disjonctive sur
// chaque machine, et minimisation du makespan (Cmax).
fn solve_jobshop(jobs: &[(&str, Vec<(&str, i64)>)], machine_names: &[&str])
    -> (Vec<Vec<(String, i64, i64)>>, i64) {
    let opt = Optimize::new();
    // Borne superieure sure : tout execute a la suite.
    let horizon: i64 = jobs.iter().flat_map(|(_, ops)| ops.iter().map(|(_, d)| *d)).sum();

    // Variables de debut : s_{j}_{k}
    let mut starts: Vec<Vec<Int>> = Vec::new();
    for (j, ops) in jobs.iter() {
        let mut row: Vec<Int> = Vec::new();
        for k in 0..ops.len() {
            row.push(Int::new_const(format!("s_{}_{}", j, k)));
        }
        starts.push(row);
    }
    let cmax = Int::new_const("cmax");

    // (1) Domaine, precedence intra-job, borne de makespan.
    for (ji, (_j, ops)) in jobs.iter().enumerate() {
        for k in 0..ops.len() {
            let d = ops[k].1;
            opt.assert(&starts[ji][k].ge(&Int::from_i64(0)));
            opt.assert(&Int::add(&[starts[ji][k].clone(), Int::from_i64(d)]).le(&cmax));
            if k > 0 {
                let pd = ops[k - 1].1;
                opt.assert(&starts[ji][k].ge(
                    &Int::add(&[starts[ji][k - 1].clone(), Int::from_i64(pd)])));
            }
        }
    }

    // (2) Exclusion disjonctive : deux operations sur la MEME machine ne se
    //     chevauchent pas -- l'une finit avant que l'autre commence.
    for m in machine_names {
        let mut ops_on_m: Vec<(usize, usize, i64)> = Vec::new();
        for (ji, (_j, ops)) in jobs.iter().enumerate() {
            for (k, (mm, d)) in ops.iter().enumerate() {
                if mm == m {
                    ops_on_m.push((ji, k, *d));
                }
            }
        }
        for a in 0..ops_on_m.len() {
            for b in (a + 1)..ops_on_m.len() {
                let (ja, ka, da) = ops_on_m[a];
                let (jb, kb, db) = ops_on_m[b];
                let disj = Bool::or(&[
                    &Int::add(&[starts[ja][ka].clone(), Int::from_i64(da)]).le(&starts[jb][kb]),
                    &Int::add(&[starts[jb][kb].clone(), Int::from_i64(db)]).le(&starts[ja][ka]),
                ]);
                opt.assert(&disj);
            }
        }
    }

    // (3) Objectif : minimiser le makespan.
    opt.assert(&cmax.le(&Int::from_i64(horizon)));
    opt.minimize(&cmax);

    match opt.check(&[]) {
        SatResult::Sat => {
            let model = match opt.get_model() {
                Some(m) => m,
                None => {
                    println!("Modele optimal introuvable.");
                    return (Vec::new(), horizon);
                }
            };
            let mut schedule: Vec<Vec<(String, i64, i64)>> = Vec::new();
            for (ji, (_j, ops)) in jobs.iter().enumerate() {
                let mut row: Vec<(String, i64, i64)> = Vec::new();
                for k in 0..ops.len() {
                    let v = model
                        .eval(&starts[ji][k], true)
                        .and_then(|x| x.as_i64())
                        .unwrap_or(0);
                    row.push((ops[k].0.to_string(), v, ops[k].1));
                }
                schedule.push(row);
            }
            let c = model.eval(&cmax, true).and_then(|x| x.as_i64()).unwrap_or(horizon);
            (schedule, c)
        }
        other => {
            println!("Instance insatisfiable : {:?}", other);
            (Vec::new(), 0)
        }
    }
}

let (greedy_schedule, greedy_cmax) = greedy_fifo(&JOBS, &MACHINES);
let (opt_schedule, opt_cmax) = solve_jobshop(&JOBS, &MACHINES);
println!("Z3 optimal : Cmax = {}h  (glouton = {}h, gain = {}h)",
    opt_cmax, greedy_cmax, greedy_cmax - opt_cmax);

Z3 optimal : Cmax = 8h  (glouton = 14h, gain = 6h)


## Anatomie du modele genere

Avant de lire le resultat, comptons ce que Z3 manipule reellement pour cette instance :

| Element | Quantite | Detail |
|---|---|---|
| Variables entieres | 7 | $s_{j,k}$ pour 6 operations + $C_{\max}$ |
| Precedences intra-job | 3 | une par job (l'operation 2 suit l'operation 1) |
| Bornes de domaine | 6 | $s_{j,k} \ge 0$, plus $s_{j,k} + d_{j,k} \le C_{\max}$ |
| Contraintes disjonctives | 6 | 3 paires sur M0, 3 paires sur M1 |

Les paires disjonctives se repartissent ainsi : M0 porte les operations J0-1, J1-2 et J2-1, soit $\binom{3}{2} = 3$ paires ; M1 porte J0-2, J1-1 et J2-2, soit 3 paires egalement. Chaque paire produit une disjonction : l'une des deux operations finit avant que l'autre commence. Ces 6 disjonctions sont le coeur combinatoire du modele — c'est elles qui codent le choix de l'ordre sur chaque machine, sans jamais l'enumerer.

Notez enfin le role d'`Optimize` (et non `Solver`) : la ou un `Solver` renverrait le **premier** ordonnancement faisable trouve, `Optimize.minimize(cmax)` exige le makespan **minimal**, avec garantie d'optimalite — le solveur prouve qu'aucune valeur inferieure n'est satisfiable. C'est une preuve, pas une heuristique.

## Resultat optimal

Le solveur trouve l'ordonnancement de makespan **minimal**. Comparons le detail op par op : la ou le glouton sequentialisait trop, Z3 entrelace les jobs (une operation d'un job pendant le creux d'un autre) pour reduire le temps total.

In [5]:
use z3::ast::{Ast, Bool, Int};
use z3::{Optimize, SatResult};

const MACHINES: [&str; 2] = ["M0", "M1"];
// Le solveur est genererique : une operation est un couple (machine, duree)
// et le nombre d'operations par job est porte par un Vec. En Python, la meme
// information est une liste et la generalisation est gratuite ; en Rust, la
// longueur d'un tableau est dans le type -- c'est la seule generalisation que
// le portage exige (l'algorithme, lui, boucle deja sur ops.len()).
let JOBS: Vec<(&str, Vec<(&str, i64)>)> = vec![
    ("J0", vec![("M0", 3), ("M1", 3)]),
    ("J1", vec![("M1", 2), ("M0", 2)]),
    ("J2", vec![("M0", 2), ("M1", 2)]),
];
// Ordonnancement glouton FIFO : chaque job dans l'ordre, chaque operation ASAP.
// Une machine n'est jamais liberee avant la fin de l'operation qui l'occupe.
fn greedy_fifo(jobs: &[(&str, Vec<(&str, i64)>)], machine_names: &[&str])
    -> (Vec<Vec<(String, i64, i64)>>, i64) {
    let mut machine_free: Vec<(String, i64)> =
        machine_names.iter().map(|m| (m.to_string(), 0)).collect();
    let mut schedule: Vec<Vec<(String, i64, i64)>> = Vec::new();
    for (_j, ops) in jobs.iter() {
        let mut job_ops: Vec<(String, i64, i64)> = Vec::new();
        for (m, d) in ops.iter() {
            // L'operation du job ne peut pas demarrer avant la fin de la precedente.
            let job_ready = job_ops.last().map(|(_, s, dur)| s + dur).unwrap_or(0);
            let start = match machine_free.iter_mut().find(|(name, _)| name == m) {
                Some(slot) => {
                    let s = std::cmp::max(slot.1, job_ready);
                    slot.1 = s + d;
                    s
                }
                None => job_ready,
            };
            job_ops.push((m.to_string(), start, *d));
        }
        schedule.push(job_ops);
    }
    let cmax = schedule.iter().flatten().map(|(_, s, d)| s + d).max().unwrap_or(0);
    (schedule, cmax)
}
// Modele Z3 du job-shop : precedence intra-job, exclusion disjonctive sur
// chaque machine, et minimisation du makespan (Cmax).
fn solve_jobshop(jobs: &[(&str, Vec<(&str, i64)>)], machine_names: &[&str])
    -> (Vec<Vec<(String, i64, i64)>>, i64) {
    let opt = Optimize::new();
    // Borne superieure sure : tout execute a la suite.
    let horizon: i64 = jobs.iter().flat_map(|(_, ops)| ops.iter().map(|(_, d)| *d)).sum();

    // Variables de debut : s_{j}_{k}
    let mut starts: Vec<Vec<Int>> = Vec::new();
    for (j, ops) in jobs.iter() {
        let mut row: Vec<Int> = Vec::new();
        for k in 0..ops.len() {
            row.push(Int::new_const(format!("s_{}_{}", j, k)));
        }
        starts.push(row);
    }
    let cmax = Int::new_const("cmax");

    // (1) Domaine, precedence intra-job, borne de makespan.
    for (ji, (_j, ops)) in jobs.iter().enumerate() {
        for k in 0..ops.len() {
            let d = ops[k].1;
            opt.assert(&starts[ji][k].ge(&Int::from_i64(0)));
            opt.assert(&Int::add(&[starts[ji][k].clone(), Int::from_i64(d)]).le(&cmax));
            if k > 0 {
                let pd = ops[k - 1].1;
                opt.assert(&starts[ji][k].ge(
                    &Int::add(&[starts[ji][k - 1].clone(), Int::from_i64(pd)])));
            }
        }
    }

    // (2) Exclusion disjonctive : deux operations sur la MEME machine ne se
    //     chevauchent pas -- l'une finit avant que l'autre commence.
    for m in machine_names {
        let mut ops_on_m: Vec<(usize, usize, i64)> = Vec::new();
        for (ji, (_j, ops)) in jobs.iter().enumerate() {
            for (k, (mm, d)) in ops.iter().enumerate() {
                if mm == m {
                    ops_on_m.push((ji, k, *d));
                }
            }
        }
        for a in 0..ops_on_m.len() {
            for b in (a + 1)..ops_on_m.len() {
                let (ja, ka, da) = ops_on_m[a];
                let (jb, kb, db) = ops_on_m[b];
                let disj = Bool::or(&[
                    &Int::add(&[starts[ja][ka].clone(), Int::from_i64(da)]).le(&starts[jb][kb]),
                    &Int::add(&[starts[jb][kb].clone(), Int::from_i64(db)]).le(&starts[ja][ka]),
                ]);
                opt.assert(&disj);
            }
        }
    }

    // (3) Objectif : minimiser le makespan.
    opt.assert(&cmax.le(&Int::from_i64(horizon)));
    opt.minimize(&cmax);

    match opt.check(&[]) {
        SatResult::Sat => {
            let model = match opt.get_model() {
                Some(m) => m,
                None => {
                    println!("Modele optimal introuvable.");
                    return (Vec::new(), horizon);
                }
            };
            let mut schedule: Vec<Vec<(String, i64, i64)>> = Vec::new();
            for (ji, (_j, ops)) in jobs.iter().enumerate() {
                let mut row: Vec<(String, i64, i64)> = Vec::new();
                for k in 0..ops.len() {
                    let v = model
                        .eval(&starts[ji][k], true)
                        .and_then(|x| x.as_i64())
                        .unwrap_or(0);
                    row.push((ops[k].0.to_string(), v, ops[k].1));
                }
                schedule.push(row);
            }
            let c = model.eval(&cmax, true).and_then(|x| x.as_i64()).unwrap_or(horizon);
            (schedule, c)
        }
        other => {
            println!("Instance insatisfiable : {:?}", other);
            (Vec::new(), 0)
        }
    }
}

let (greedy_schedule, greedy_cmax) = greedy_fifo(&JOBS, &MACHINES);
let (opt_schedule, opt_cmax) = solve_jobshop(&JOBS, &MACHINES);

println!("Comparaison glouton vs Z3 optimal :\n");
for (ji, (j, _ops)) in JOBS.iter().enumerate() {
    let g: Vec<String> = greedy_schedule[ji]
        .iter()
        .map(|(m, s, d)| format!("{}[{},{}]", m, s, s + d))
        .collect();
    let o: Vec<String> = opt_schedule[ji]
        .iter()
        .map(|(m, s, d)| format!("{}[{},{}]", m, s, s + d))
        .collect();
    println!("  {}", j);
    println!("    glouton : {}", g.join(", "));
    println!("    optimal : {}", o.join(", "));
}
println!("\nMakespan : glouton {}h -> optimal {}h (gain de {}h, {}%)",
    greedy_cmax,
    opt_cmax,
    greedy_cmax - opt_cmax,
    (100.0 * (greedy_cmax - opt_cmax) as f64 / greedy_cmax as f64).round() as i64);

Comparaison glouton vs Z3 optimal :


  J0


    glouton : M0[0,3], M1[3,6]


    optimal : M0[0,3], M1[3,6]


  J1


    glouton : M1[6,8], M0[8,10]


    optimal : M1[1,3], M0[6,8]


  J2


    glouton : M0[10,12], M1[12,14]


    optimal : M0[4,6], M1[6,8]


Makespan : glouton 14h -> optimal 8h (gain de 6h, 43%)


## Lecture de la comparaison : d'ou vient l'ecart 14 h -> 8 h ?

La sortie ci-dessus se lit ligne par ligne, et tout se joue sur **J1**. J0 est identique dans les deux ordonnancements (M0[0,3] puis M1[3,6]) : c'est le premier job traite, le glouton ne peut pas se tromper. Mais observez J1 :

- **Glouton** : M1[6,8] puis M0[8,10]. Pourquoi attendre t = 6 pour sa premiere operation ? Parce que le glouton traite les jobs **les uns apres les autres** : quand vient le tour de J1, la plage M1[3,6] est deja reservee par J0, et le glouton ne place jamais une operation **dans un trou** laisse avant un placement precedent. La fenetre M1[0,3] — parfaitement libre a ce moment-la — reste inutilisee.
- **Optimal** : Z3 n'a aucune notion d'ordre de traitement : il place les 6 operations simultanement dans le temps, et remplit la fenetre M1 que le glouton a laissee vide.

J2 subit le meme effet en cascade : recule par le glouton, il est avance par l'optimal. Le glouton aboutit a 14 h — exactement l'horizon sequentiel, la pire borne possible — tandis que l'optimal gagne 6 h (**43 %**), une heure seulement au-dessus de la borne de charge (7 h).

**Note sur la comparaison des modeles.** Le makespan est unique (8 h), mais l'ordonnancement optimal ne l'est pas : plusieurs placements differentes atteignent 8 h, et Z3 peut en choisir un autre que le jumeau Python. C'est le **makespan** qui est compare entre les deux carnets, pas le placement op par op.

## Visualisation : diagramme de Gantt

Le diagramme de Gantt rend visible la difference : chaque ligne est une machine, chaque barre coloree une operation. Le glouton (haut) laisse des trous ; l'optimal Z3 (bas) tasse les operations pour finir plus tot.

**Ce qui change en Rust** : le jumeau Python produit une figure `matplotlib` (image PNG). Ce carnet n'embarque aucune bibliotheque de tracage — le crate `z3` n'a rien a voir avec le rendu, et ajouter un moteur de figures serait hors sujet. Le diagramme est donc rendu **en texte** : une ligne par machine, une colonne par heure, chaque barre portant le nom du job. C'est la meme information, dans un rendu degrade assume.

In [6]:
use z3::ast::{Ast, Bool, Int};
use z3::{Optimize, SatResult};

const MACHINES: [&str; 2] = ["M0", "M1"];
// Le solveur est genererique : une operation est un couple (machine, duree)
// et le nombre d'operations par job est porte par un Vec. En Python, la meme
// information est une liste et la generalisation est gratuite ; en Rust, la
// longueur d'un tableau est dans le type -- c'est la seule generalisation que
// le portage exige (l'algorithme, lui, boucle deja sur ops.len()).
let JOBS: Vec<(&str, Vec<(&str, i64)>)> = vec![
    ("J0", vec![("M0", 3), ("M1", 3)]),
    ("J1", vec![("M1", 2), ("M0", 2)]),
    ("J2", vec![("M0", 2), ("M1", 2)]),
];
// Ordonnancement glouton FIFO : chaque job dans l'ordre, chaque operation ASAP.
// Une machine n'est jamais liberee avant la fin de l'operation qui l'occupe.
fn greedy_fifo(jobs: &[(&str, Vec<(&str, i64)>)], machine_names: &[&str])
    -> (Vec<Vec<(String, i64, i64)>>, i64) {
    let mut machine_free: Vec<(String, i64)> =
        machine_names.iter().map(|m| (m.to_string(), 0)).collect();
    let mut schedule: Vec<Vec<(String, i64, i64)>> = Vec::new();
    for (_j, ops) in jobs.iter() {
        let mut job_ops: Vec<(String, i64, i64)> = Vec::new();
        for (m, d) in ops.iter() {
            // L'operation du job ne peut pas demarrer avant la fin de la precedente.
            let job_ready = job_ops.last().map(|(_, s, dur)| s + dur).unwrap_or(0);
            let start = match machine_free.iter_mut().find(|(name, _)| name == m) {
                Some(slot) => {
                    let s = std::cmp::max(slot.1, job_ready);
                    slot.1 = s + d;
                    s
                }
                None => job_ready,
            };
            job_ops.push((m.to_string(), start, *d));
        }
        schedule.push(job_ops);
    }
    let cmax = schedule.iter().flatten().map(|(_, s, d)| s + d).max().unwrap_or(0);
    (schedule, cmax)
}
// Modele Z3 du job-shop : precedence intra-job, exclusion disjonctive sur
// chaque machine, et minimisation du makespan (Cmax).
fn solve_jobshop(jobs: &[(&str, Vec<(&str, i64)>)], machine_names: &[&str])
    -> (Vec<Vec<(String, i64, i64)>>, i64) {
    let opt = Optimize::new();
    // Borne superieure sure : tout execute a la suite.
    let horizon: i64 = jobs.iter().flat_map(|(_, ops)| ops.iter().map(|(_, d)| *d)).sum();

    // Variables de debut : s_{j}_{k}
    let mut starts: Vec<Vec<Int>> = Vec::new();
    for (j, ops) in jobs.iter() {
        let mut row: Vec<Int> = Vec::new();
        for k in 0..ops.len() {
            row.push(Int::new_const(format!("s_{}_{}", j, k)));
        }
        starts.push(row);
    }
    let cmax = Int::new_const("cmax");

    // (1) Domaine, precedence intra-job, borne de makespan.
    for (ji, (_j, ops)) in jobs.iter().enumerate() {
        for k in 0..ops.len() {
            let d = ops[k].1;
            opt.assert(&starts[ji][k].ge(&Int::from_i64(0)));
            opt.assert(&Int::add(&[starts[ji][k].clone(), Int::from_i64(d)]).le(&cmax));
            if k > 0 {
                let pd = ops[k - 1].1;
                opt.assert(&starts[ji][k].ge(
                    &Int::add(&[starts[ji][k - 1].clone(), Int::from_i64(pd)])));
            }
        }
    }

    // (2) Exclusion disjonctive : deux operations sur la MEME machine ne se
    //     chevauchent pas -- l'une finit avant que l'autre commence.
    for m in machine_names {
        let mut ops_on_m: Vec<(usize, usize, i64)> = Vec::new();
        for (ji, (_j, ops)) in jobs.iter().enumerate() {
            for (k, (mm, d)) in ops.iter().enumerate() {
                if mm == m {
                    ops_on_m.push((ji, k, *d));
                }
            }
        }
        for a in 0..ops_on_m.len() {
            for b in (a + 1)..ops_on_m.len() {
                let (ja, ka, da) = ops_on_m[a];
                let (jb, kb, db) = ops_on_m[b];
                let disj = Bool::or(&[
                    &Int::add(&[starts[ja][ka].clone(), Int::from_i64(da)]).le(&starts[jb][kb]),
                    &Int::add(&[starts[jb][kb].clone(), Int::from_i64(db)]).le(&starts[ja][ka]),
                ]);
                opt.assert(&disj);
            }
        }
    }

    // (3) Objectif : minimiser le makespan.
    opt.assert(&cmax.le(&Int::from_i64(horizon)));
    opt.minimize(&cmax);

    match opt.check(&[]) {
        SatResult::Sat => {
            let model = match opt.get_model() {
                Some(m) => m,
                None => {
                    println!("Modele optimal introuvable.");
                    return (Vec::new(), horizon);
                }
            };
            let mut schedule: Vec<Vec<(String, i64, i64)>> = Vec::new();
            for (ji, (_j, ops)) in jobs.iter().enumerate() {
                let mut row: Vec<(String, i64, i64)> = Vec::new();
                for k in 0..ops.len() {
                    let v = model
                        .eval(&starts[ji][k], true)
                        .and_then(|x| x.as_i64())
                        .unwrap_or(0);
                    row.push((ops[k].0.to_string(), v, ops[k].1));
                }
                schedule.push(row);
            }
            let c = model.eval(&cmax, true).and_then(|x| x.as_i64()).unwrap_or(horizon);
            (schedule, c)
        }
        other => {
            println!("Instance insatisfiable : {:?}", other);
            (Vec::new(), 0)
        }
    }
}
// Diagramme de Gantt en texte : une ligne par machine, une colonne par heure.
// (Le jumeau Python produit une figure matplotlib ; ce carnet n'a pas de
//  bibliotheque de tracage -- le carnet rend la meme information en ASCII.)
fn gantt_lines(schedule: &[Vec<(String, i64, i64)>],
               jobs: &[(&str, Vec<(&str, i64)>)],
               machines: &[&str],
               xmax: i64) -> Vec<String> {
    let mut lines: Vec<String> = Vec::new();
    for m in machines {
        let mut ligne = format!("  {} |", m);
        for t in 0..=xmax {
            let mut barre = String::from(" ");
            for (ji, ops) in schedule.iter().enumerate() {
                for (mach, s, d) in ops.iter() {
                    if mach.as_str() == *m && *s <= t && t < *s + *d {
                        if barre == " " {
                            barre = jobs[ji].0.to_string();
                        } else {
                            barre = format!("{}+", barre);
                        }
                    }
                }
            }
            ligne.push_str(&format!("{}|", barre));
        }
        lines.push(ligne);
    }
    lines
}

let (greedy_schedule, greedy_cmax) = greedy_fifo(&JOBS, &MACHINES);
let (opt_schedule, opt_cmax) = solve_jobshop(&JOBS, &MACHINES);
let xmax = std::cmp::max(greedy_cmax, opt_cmax);

println!("Gantt glouton (Cmax = {}h) :", greedy_cmax);
for ligne in gantt_lines(&greedy_schedule, &JOBS, &MACHINES, xmax) {
    println!("{}", ligne);
}
println!("Gantt Z3 optimal (Cmax = {}h) :", opt_cmax);
for ligne in gantt_lines(&opt_schedule, &JOBS, &MACHINES, xmax) {
    println!("{}", ligne);
}
println!("Diagramme de Gantt : glouton (haut) vs optimal (bas).");
// evcxr afficherait sinon la valeur unitaire () du bloc : on neutralise
// l'affichage pour que la sortie reste comparable a celle du jumeau Python.
let _ = &opt_schedule;


Gantt glouton (Cmax = 14h) :


  M0 |J0|J0|J0| | | | | |J1|J1|J2|J2| | | |


  M1 | | | |J0|J0|J0|J1|J1| | | | |J2|J2| |


Gantt Z3 optimal (Cmax = 8h) :


  M0 |J0|J0|J0| |J2|J2|J1|J1| | | | | | | |


  M1 | |J1|J1|J0|J0|J0|J2|J2| | | | | | | |


Diagramme de Gantt : glouton (haut) vs optimal (bas).


## Lecture du diagramme

Les deux Gantt ci-dessus rendent visibles les defauts du glouton. Sur le Gantt optimal (bas), chaque machine ne reste **inoccupee qu'une seule heure** : M0 sur [3,4] et M1 sur [0,1]. Les trois jobs s'entrelacent — une operation d'un job pendant le creux d'un autre — et le makespan de 8 h correspond a la fin simultanee des deux dernieres operations.

Un point merite reflexion : la borne inferieure de charge (7 h) n'est **pas atteinte**. Un ordonnancement a 7 h remplirait chaque machine sans aucune interruption — le solveur a explore cette possibilite et prouve qu'elle viole necessairement une precedence ou une disjonction. C'est l'apport decisif d'`Optimize` : la valeur 8 h est **certifiee minimale**. Un metaheuristique (recuit simule, colonies de fourmis) pourrait trouver 8 h aussi, mais ne pourrait jamais certifier qu'il n'existe pas de solution a 7 h.

Les exercices suivants prolongent exactement ce geste : ajouter un job (l'espace de recherche grandit), une deadline (contrainte dure), une machine (nouvelles disjonctions).

## Exercices

Les trois exercices suivants vous font manipuler le modele : etendre l'instance (1), ajouter une contrainte metier de deadline (2), et monter en dimension avec une troisieme machine (3). Chaque stub reutilise `solve_jobshop` defini plus haut.

### Exercice 1 — Ajout d'un quatrieme job

**Objectif** : ajouter un job `J3 = [("M1", 1), ("M0", 3)]` (1h sur M1 puis 3h sur M0) et re-resoudre. Observez comment le makespan optimal augmente (ou non) et comment Z3 re-entrelace les operations.

**Indices** : copiez la constante `JOBS` dans `JOBS_4` avec l'entree J3 supplementaire ; appelez `solve_jobshop(&JOBS_4, &MACHINES)`.

In [7]:
// Exercice 1 : ajoutez un quatrieme job J3 et re-resolvez.
// TODO etudiant :
// Etape 1 : copiez JOBS dans JOBS_4 avec l'entree J3 = ("J3", [("M1", 1), ("M0", 3)]).
// Etape 2 : resolvez via solve_jobshop(&JOBS_4, &MACHINES).
// Etape 3 : comparez le nouveau Cmax a l'optimum 3-jobs (opt_cmax).
let cmax_4_jobs: Option<i64> = None; // TODO etudiant
let _ = cmax_4_jobs;
println!("Exercice 1 a completer : ajouter un 4eme job et mesurer le nouveau Cmax optimal.");

Exercice 1 a completer : ajouter un 4eme job et mesurer le nouveau Cmax optimal.


### Exercice 2 — Contrainte d'urgence (deadline)

**Objectif** : le job `J1` vient de passer urgent : imposez qu'il termine avant $t = 6$ (dans l'ordonnancement optimal exhibe plus haut, J1 termine a $t = 8$). Ajoutez cette contrainte au modele et verifiez que le makespan optimal augmente — mesure au solveur, Cmax passe de 8 a 9 : la deadline force un ordonnancement moins efficace.

**Indices** : la fin de J1 est `s_1_1 + duree_derniere_op` ; ajoutez la contrainte `opt.assert(...)` avant `opt.minimize`. Dupliquez `solve_jobshop` en `solve_jobshop_deadline`.

In [8]:
// Exercice 2 : ajoutez une deadline (J1 doit terminer avant t=6).
// TODO etudiant :
// Etape 1 : copiez solve_jobshop en solve_jobshop_deadline(jobs, machines, deadline_j1 = 6).
// Etape 2 : ajoutez la contrainte de fin de J1 avant opt.minimize.
// Etape 3 : resolvez et comparez le Cmax a opt_cmax (sans deadline).
let cmax_avec_deadline: Option<i64> = None; // TODO etudiant
let _ = cmax_avec_deadline;
println!("Exercice 2 a completer : mesurer le surcout d'une deadline sur J1.");

Exercice 2 a completer : mesurer le surcout d'une deadline sur J1.


### Exercice 3 — Extension a 3 machines

**Objectif** : etendre l'instance a 3 machines (M0, M1, M2) avec des jobs a 3 operations chacun, et verifiez que `solve_jobshop` generalise sans modification de l'algorithme (le solveur est deja generique : une operation est un couple `(machine, duree)` et le nombre d'operations par job vit dans un `Vec`).

**Indices** : definissez `MACHINES_3` (trois noms de machines) et des `JOBS_3` ou chaque job a 3 operations sur des machines distinctes ; la fonction `solve_jobshop` gere deja le cas general.

**Ce que le portage a change, et pourquoi** : en Python, une instance est une liste -- passer de 2 a 3 operations ne change rien au type. En Rust, la longueur d'un tableau est dans le type (`[(&str, i64); 2]`), si bien que le portage a generalise une fois pour toutes la signature en `Vec<(&str, i64)>` : c'est la seule difference de forme entre les deux jumeaux, l'algorithme et les sorties sont identiques.

In [9]:
// Exercice 3 : etendez l'instance a 3 machines, 3 operations par job.
// TODO etudiant :
// Etape 1 : definissez JOBS_3 (3 jobs, chacun 3 operations sur des machines distinctes)
//           sous la forme Vec<(&str, Vec<(&str, i64)>)> -- le solveur est deja
//           generique, il accepte n'importe quel nombre d'operations par job.
// Etape 2 : resolvez via solve_jobshop(&JOBS_3, &MACHINES_3).
// Question : le temps de resolution grimpe-t-il avec 3 machines ? Mesurez-le.
let cmax_3_machines: Option<i64> = None; // TODO etudiant
let _ = cmax_3_machines;
println!("Exercice 3 a completer : generaliser a 3 machines et observer le temps de resolution.");

Exercice 3 a completer : generaliser a 3 machines et observer le temps de resolution.


## Conclusion

Le job-shop scheduling illustre le gain du **declaratif** sur une classe de problemes ou l'heuristique gloutonne est structurellement sous-optimale :

- La **contrainte disjonctive** `Or(s_a + d_a <= s_b, s_b + d_b <= s_a)` (exclusion mutuelle sur une machine) est difficile a coder en imperatif sans enumerer tous les ordres possibles ; Z3 la traite nativement.
- `Optimize.minimize` trouve le makespan **optimal** (prouvablement minimal), la ou FIFO se contente d'un ordonnancement realisable mais myope.
- Le modele est **parametre par les donnees** : ajouter un job, une machine ou une deadline se fait en modifiant l'instance ou en ajoutant une contrainte, sans re-ecrire l'algorithme de resolution.

**Limites** : sur de tres grandes instances (dizaines de jobs × dizaines de machines), le solveur peut devenir lent — c'est le prix de l'optimalite garantie. Dans ce cas, on combine souvent Z3 avec des heuristiques (fournir un ordonnancement glouton comme borne superieure aide le solveur a pruner).

**Portage Rust** : l'API du crate `z3` est plus explicite que celle du module Python — chaque operateur est une fonction (`Int::add`, `Bool::or`), le modele s'obtient par `get_model()` puis `eval()`. Le prix a payer est la redaction des types, mais le modele lui-meme est rigoureusement le meme : les memes grandeurs (Cmax glouton 14 h, Cmax optimal 8 h) sortent des deux cotes.

Suite logique : retour a la [serie Z3-API](README.md), ou comparez avec le jumeau Python [Z3-08-Ordonnancement-Python](Z3-08-Ordonnancement-Python.ipynb).